# SPECTRA DP0.2 Smoke Test

This notebook runs a small Rubin DP0.2/DC2 cone-search test through SPECTRA. It starts from the curated `example_configs/config_dp02_test.yaml`, writes a notebook-local copy with a small object cap, runs the pipeline, and inspects the resulting `fit_summary.csv`.

## 1. Set Up the Repository Path

Run this notebook from the repository root or from the `notebooks/` directory.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
if not (repo_root / "src").exists():
    raise RuntimeError("Run this notebook from the SPECTRA repository root or from notebooks/.")

sys.path.insert(0, str(repo_root))
repo_root

## 2. Imports and Rubin Token

The Rubin query layer reads `RSP_TOKEN` from the environment. If it is missing, the next cell asks for it without echoing it back into the notebook.

In [ ]:
import getpass
import yaml
import pandas as pd

from src.cli import validate_config
from src.main import main
from src.data.rubin_query import RubinDataQuery

if not os.environ.get("RSP_TOKEN"):
    os.environ["RSP_TOKEN"] = getpass.getpass("Rubin RSP token: ")

print("RSP_TOKEN is available for this notebook session.")

## 3. Prepare a Small DP0.2 Config

This cell copies the public DP0.2 example config and makes the run intentionally small. The generated config is ignored by Git and can be changed freely.

In [ ]:
base_config_path = repo_root / "example_configs" / "config_dp02_test.yaml"
with base_config_path.open("r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

config["input"]["max_objects"] = 2
config["input"]["radius_arcsec"] = 30.0
config["plotting"]["output_dir"] = str(repo_root / "outputs" / "notebook_dp02_smoke")
config["plotting"]["show_plots"] = False
config["fitting"]["method"] = "ml"

notebook_config_path = repo_root / "rsp_config.yaml"
notebook_config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

print(notebook_config_path)
config

## 4. Validate the Config

This catches missing required sections before sending a query to Rubin.

In [ ]:
status = validate_config(str(notebook_config_path))
if status != 0:
    raise RuntimeError(f"Config validation failed with status {status}")

print("Config validation passed.")

## 5. Preview the DP0.2 Query

This quick preview checks that the RSP token and DP0.2 cone search work before running the fitter.

In [ ]:
query = RubinDataQuery(token=os.environ["RSP_TOKEN"])
preview = query.cone_search(
    ra=config["input"]["ra"],
    dec=config["input"]["dec"],
    radius_arcsec=config["input"]["radius_arcsec"],
    flux_type=config["rubin"].get("flux_type", "cModelFlux"),
    bands=config["rubin"].get("bands"),
    max_objects=config["input"]["max_objects"],
)

print(f"Loaded {len(preview)} DP0.2 object(s).")
for object_id, phot_data in preview:
    print(object_id, phot_data.get("bands"), phot_data["obs_flux"].shape)

## 6. Run SPECTRA

This runs maximum-likelihood fitting on the same small DP0.2 selection and writes plots plus a summary table under `outputs/notebook_dp02_smoke/`.

In [ ]:
main(str(notebook_config_path))

## 7. Inspect Fit Summary

The table below is the first pass quality check. Pay closest attention to `chi2_red`, parameter values at prior boundaries, and whether any object produced missing values.

In [ ]:
summary_path = Path(config["plotting"]["output_dir"]) / "fit_summary.csv"
summary = pd.read_csv(summary_path)
summary

## 8. Quick Diagnostics

This cell flags high reduced chi-square values and lists the per-object output folders.

In [ ]:
display_cols = [col for col in ["object_id", "redshift", "chi2_red", "mass", "age", "metallicity", "dust"] if col in summary.columns]
display(summary[display_cols])

poor = summary.loc[summary["chi2_red"] > 2, display_cols]
if len(poor):
    print("Objects with chi2_red > 2; inspect residual plots before using these scientifically:")
    display(poor)
else:
    print("All objects have chi2_red <= 2 in this smoke test.")

for path in sorted(Path(config["plotting"]["output_dir"]).glob("*")):
    if path.is_dir():
        print(path.relative_to(repo_root))